# 03 - Fine-tune DeBERTa-v3 for aspect sentiment

Fine-tunes `microsoft/deberta-v3-base` to read an (aspect, sentence) pair and predict negative / neutral / positive / conflict. This is the strongest model in the project and the one the app uses.

**Input:** `data/splits/train_split.csv` and `validation_split.csv` (made by notebook 00)
**Output:** the model in `models/deberta_v3_absa/` (weights, tokenizer, label map, metrics, training history)

The test set is not used here. It is scored once, later, in notebook 05, together with the other two models. Validation rows pick the best epoch.

Training takes a while on a laptop GPU. The Trainer's own per-epoch checkpoints go to `models/deberta_v3_runs/` and can be deleted afterwards.

In [1]:
import os

# Set before huggingface_hub is imported. Turns off the Xet downloader, which can stall on some networks.
os.environ["HF_HUB_DISABLE_XET"] = "1"

import json
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from huggingface_hub import snapshot_download
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

from opinion_owl.config import (
    DEBERTA_BASE_DIR, DEBERTA_DIR, DEBERTA_RUNS_DIR, TRAIN_SPLIT, VALIDATION_SPLIT,
)

## 1. Settings

In [2]:
BASE_MODEL_ID = "microsoft/deberta-v3-base"
MODEL_NAME = str(DEBERTA_BASE_DIR)  # local copy, downloaded once below

SEED = 42
MAX_LENGTH = 128

# Safe starting point for a 6 GB laptop GPU. Effective batch size = 4 x 4 = 16.
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 4

LEARNING_RATE = 2e-5
NUM_EPOCHS = 5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
EARLY_STOPPING_PATIENCE = 2  # stop if validation macro-F1 does not improve for 2 epochs

LABEL_TO_ID = {"negative": 0, "neutral": 1, "positive": 2, "conflict": 3}
ID_TO_LABEL = {label_id: label for label, label_id in LABEL_TO_ID.items()}
NUM_LABELS = len(LABEL_TO_ID)

# Download the pretrained model once; later runs skip this.
if not (DEBERTA_BASE_DIR / "pytorch_model.bin").exists():
    snapshot_download(
        BASE_MODEL_ID,
        local_dir=str(DEBERTA_BASE_DIR),
        allow_patterns=["config.json", "spm.model", "pytorch_model.bin"],
    )
print("Base model files:", sorted(p.name for p in DEBERTA_BASE_DIR.iterdir() if p.is_file()))

Base model files: ['config.json', 'pytorch_model.bin', 'spm.model', 'tokenizer_config.json']


In [3]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_FP16 = torch.cuda.is_available()  # half precision: faster and lighter on GPU memory

print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("WARNING: CUDA unavailable. DeBERTa training on CPU will be very slow.")

Device: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
GPU memory: 6.0 GB


## 2. Load the splits

In [4]:
# Notebook 00 makes these files once, so every model is trained on the same rows
# and the test rows are never seen during training.
missing = [p for p in (TRAIN_SPLIT, VALIDATION_SPLIT) if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Missing split file(s):\n" + "\n".join(str(p) for p in missing)
        + "\n\nRun notebook 00 (prepare data) first; it creates the splits."
    )


def load_split(path):
    df = pd.read_csv(path).dropna(subset=["text", "aspect_term", "polarity"])
    df["label"] = df["polarity"].map(LABEL_TO_ID)
    return df.dropna(subset=["label"]).astype({"label": int}).reset_index(drop=True)


train_df = load_split(TRAIN_SPLIT)
validation_df = load_split(VALIDATION_SPLIT)

print("Train:", len(train_df), "| Validation:", len(validation_df))
print(train_df["polarity"].value_counts())

Train: 2112 | Validation: 264
polarity
positive    932
negative    780
neutral     365
conflict     35
Name: count, dtype: int64


In [5]:
# "conflict" is rare, so mistakes on it count for more during training.
# weight = total rows / (number of classes x rows in that class)
train_counts = train_df["label"].value_counts().sort_index()
class_weights = [len(train_df) / (NUM_LABELS * train_counts.get(i, 1)) for i in range(NUM_LABELS)]
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

print("Class weights (higher = rarer class):")
for class_id, weight in enumerate(class_weights):
    print(f"  {ID_TO_LABEL[class_id]:10s} {weight:.3f}")

Class weights (higher = rarer class):
  negative   0.677
  neutral    1.447
  positive   0.567
  conflict   15.086


## 3. Tokenizer, datasets and model

In [6]:
# use_fast=False avoids the sentencepiece -> fast-tokenizer conversion step, which can
# fail on some transformers versions. The app and notebooks 05 and 09 load it the same way.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

# The model reads the pair "[CLS] aspect [SEP] sentence [SEP]". truncation="only_second"
# shortens the sentence if needed and never cuts the aspect.
example = tokenizer(
    "8gb ram",
    "The laptop is great for school, but 8GB RAM is restrictive.",
    truncation="only_second",
    max_length=MAX_LENGTH,
)
print(tokenizer.convert_ids_to_tokens(example["input_ids"]))

[transformers] The tokenizer you are loading from 'C:\Programming\opinion-owl\models\deberta_v3_base_pretrained' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


['[CLS]', '▁8', 'gb', '▁ram', '[SEP]', '▁The', '▁laptop', '▁is', '▁great', '▁for', '▁school', ',', '▁but', '▁8', 'GB', '▁RAM', '▁is', '▁restrictive', '.', '[SEP]']


In [7]:
class AspectPolarityDataset(torch.utils.data.Dataset):
    def __init__(self, dataframe, tokenizer, max_length):
        self.labels = dataframe["label"].tolist()
        self.encodings = tokenizer(
            dataframe["aspect_term"].tolist(),
            dataframe["text"].tolist(),
            truncation="only_second",
            max_length=max_length,
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        item = {key: values[index] for key, values in self.encodings.items()}
        item["labels"] = self.labels[index]
        return item


train_dataset = AspectPolarityDataset(train_df, tokenizer, MAX_LENGTH)
validation_dataset = AspectPolarityDataset(validation_df, tokenizer, MAX_LENGTH)

# Pads each batch to its own longest example instead of always to MAX_LENGTH.
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

lengths = [len(x) for x in train_dataset.encodings["input_ids"]]
print("Median token length:", int(np.median(lengths)))
print("95th percentile length:", int(np.percentile(lengths, 95)))
print("Rows truncated at MAX_LENGTH:", sum(l >= MAX_LENGTH for l in lengths))

Median token length: 26
95th percentile length: 82
Rows truncated at MAX_LENGTH: 0


In [8]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID_TO_LABEL,
    label2id=LABEL_TO_ID,
    dtype=torch.float32,  # weights stay in full precision; fp16 is applied during training only
)

print("Parameters:", f"{sum(p.numel() for p in model.parameters()) / 1e6:.1f} M")

c:\Users\ujjwa\anaconda3\Lib\site-packages\torch\jit\_script.py:1485: FutureWarning: `torch.jit.script` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: C:\Programming\opinion-owl\models\deberta_v3_base_pretrained
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight              

Parameters: 184.4 M


## 4. Training setup

In [9]:
def compute_metrics(eval_prediction):
    """Run by the Trainer after every epoch on the validation set."""
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, labels=list(range(NUM_LABELS)), average="macro", zero_division=0),
    }


class WeightedLossTrainer(Trainer):
    """Trainer that uses class-weighted cross entropy, so the rare conflict class is not ignored."""

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        loss_function = nn.CrossEntropyLoss(weight=class_weights_tensor.to(logits.device))
        loss = loss_function(logits.view(-1, NUM_LABELS), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

In [10]:
# The best epoch (highest validation macro-F1) is reloaded automatically at the end.
# Note: a float below 1 for warmup_steps is read as a fraction of all training steps in
# newer transformers versions. On older versions, use warmup_ratio=WARMUP_RATIO instead.
training_args = TrainingArguments(
    output_dir=str(DEBERTA_RUNS_DIR),
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP_RATIO,
    lr_scheduler_type="linear",
    fp16=USE_FP16,
    logging_steps=25,
    report_to="none",
    seed=SEED,
    dataloader_num_workers=0,
)

trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)

print("Effective batch size:", TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)
print("Mixed precision fp16:", USE_FP16)

Effective batch size: 16
Mixed precision fp16: True


## 5. Train

If you hit `CUDA out of memory`, set `TRAIN_BATCH_SIZE = 2` and `GRADIENT_ACCUMULATION_STEPS = 8`, restart the kernel, and rerun.
If the loss becomes `nan`, set `USE_FP16 = False` (slower but more stable) or lower `LEARNING_RATE` to `1e-5`.

In [11]:
train_result = trainer.train()
print("Training finished.")
print(train_result.metrics)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,4.357169,1.059942,0.685606,0.481536
2,3.335525,0.919266,0.753788,0.526205
3,2.001035,0.981173,0.746212,0.531113
4,2.143924,1.102444,0.753788,0.533616
5,1.349052,1.147478,0.768939,0.535944


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training finished.
{'train_runtime': 1540.7475, 'train_samples_per_second': 6.854, 'train_steps_per_second': 0.428, 'total_flos': 292139811812832.0, 'train_loss': 2.791414957335501, 'epoch': 5.0}


In [12]:
# The Trainer's log, one row per logging step and per epoch. Saved with the model for later.
history = pd.DataFrame(trainer.state.log_history)
history.to_csv(DEBERTA_DIR / "deberta_training_history.csv", index=False)

if "eval_macro_f1" in history.columns:
    display(history[history["eval_macro_f1"].notna()][["epoch", "eval_loss", "eval_accuracy", "eval_macro_f1"]])

,epoch,eval_loss,eval_accuracy,eval_macro_f1
5,1.0,1.059942,0.685606,0.481536
11,2.0,0.919266,0.753788,0.526205
17,3.0,0.981173,0.746212,0.531113
24,4.0,1.102444,0.753788,0.533616
30,5.0,1.147478,0.768939,0.535944


## 6. Validation report for the best epoch

In [13]:
validation_output = trainer.predict(validation_dataset)

LABELS = [ID_TO_LABEL[i] for i in range(NUM_LABELS)]
validation_true = [ID_TO_LABEL[i] for i in validation_output.label_ids]
validation_pred = [ID_TO_LABEL[i] for i in np.argmax(validation_output.predictions, axis=-1)]

validation_accuracy = accuracy_score(validation_true, validation_pred)
validation_macro_f1 = f1_score(validation_true, validation_pred, labels=LABELS, average="macro", zero_division=0)

print("Best-epoch validation accuracy:", round(validation_accuracy, 4))
print("Best-epoch validation Macro-F1:", round(validation_macro_f1, 4))
print(classification_report(validation_true, validation_pred, labels=LABELS, zero_division=0))
print("Confusion matrix (rows = true, columns = predicted):")
print(pd.DataFrame(
    confusion_matrix(validation_true, validation_pred, labels=LABELS),
    index=[f"true_{l}" for l in LABELS],
    columns=[f"pred_{l}" for l in LABELS],
))

Best-epoch validation accuracy: 0.7689
Best-epoch validation Macro-F1: 0.5359
              precision    recall  f1-score   support

    negative       0.79      0.83      0.81        98
     neutral       0.56      0.43      0.49        46
    positive       0.82      0.88      0.85       116
    conflict       0.00      0.00      0.00         4

    accuracy                           0.77       264
   macro avg       0.54      0.54      0.54       264
weighted avg       0.75      0.77      0.76       264

Confusion matrix (rows = true, columns = predicted):
               pred_negative  pred_neutral  pred_positive  pred_conflict
true_negative             81            11              6              0
true_neutral              10            20             16              0
true_positive              8             5            102              1
true_conflict              4             0              0              0


## 7. Save the model

In [14]:
trainer.save_model(str(DEBERTA_DIR))
tokenizer.save_pretrained(str(DEBERTA_DIR))

with open(DEBERTA_DIR / "label_map.json", "w") as f:
    json.dump({"label_to_id": LABEL_TO_ID, "id_to_label": ID_TO_LABEL}, f, indent=2)

metrics_summary = {
    "model": "deberta_v3_base",
    "base_checkpoint": BASE_MODEL_ID,
    "validation_accuracy": float(validation_accuracy),
    "validation_macro_f1": float(validation_macro_f1),
    "train_rows": len(train_df),
    "validation_rows": len(validation_df),
    "max_length": MAX_LENGTH,
    "learning_rate": LEARNING_RATE,
    "effective_batch_size": TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
}
with open(DEBERTA_DIR / "training_metrics.json", "w") as f:
    json.dump(metrics_summary, f, indent=2)

print("Saved model, tokenizer, label map and metrics to:", DEBERTA_DIR)
print(metrics_summary)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model, tokenizer, label map and metrics to: C:\Programming\opinion-owl\models\deberta_v3_absa
{'model': 'deberta_v3_base', 'base_checkpoint': 'microsoft/deberta-v3-base', 'validation_accuracy': 0.7689393939393939, 'validation_macro_f1': 0.535943756825628, 'train_rows': 2112, 'validation_rows': 264, 'max_length': 128, 'learning_rate': 2e-05, 'effective_batch_size': 16}


## 8. Check the saved model

In [15]:
# Reload from disk the way the app and notebooks 05 and 09 will.
loaded_tokenizer = AutoTokenizer.from_pretrained(str(DEBERTA_DIR), use_fast=False)
loaded_model = AutoModelForSequenceClassification.from_pretrained(str(DEBERTA_DIR)).to(device).eval()


def predict_deberta_polarity(text, aspect_term):
    encoded = loaded_tokenizer(
        aspect_term.lower(), text, truncation="only_second", max_length=MAX_LENGTH, return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        probabilities = torch.softmax(loaded_model(**encoded).logits, dim=1)[0].cpu().numpy()
    predicted_id = int(np.argmax(probabilities))

    return {
        "aspect_term": aspect_term,
        "predicted_polarity": ID_TO_LABEL[predicted_id],
        "confidence": round(float(probabilities[predicted_id]), 4),
        "all_class_scores": {ID_TO_LABEL[i]: round(float(p), 4) for i, p in enumerate(probabilities)},
    }


# The reloaded model should repeat the predictions made during evaluation. A few rows
# can differ slightly because training ran in fp16 and this check runs in full precision.
check_rows = validation_df.head(40)
reloaded = [predict_deberta_polarity(r.text, r.aspect_term)["predicted_polarity"] for r in check_rows.itertuples()]
agreement = np.mean([a == b for a, b in zip(reloaded, validation_pred[:40])])
print(f"Agreement with the validation predictions on {len(check_rows)} rows: {agreement:.1%}")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Agreement with the validation predictions on 40 rows: 100.0%


In [16]:
sample_comment = (
    "Great for school, browsing and Office work, but the 8GB RAM is restrictive "
    "and AAA gaming will struggle."
)
for aspect in ["school", "productivity", "8gb ram", "gaming performance"]:
    print(predict_deberta_polarity(sample_comment, aspect))

{'aspect_term': 'school', 'predicted_polarity': 'positive', 'confidence': 0.6877, 'all_class_scores': {'negative': 0.2433, 'neutral': 0.0253, 'positive': 0.6877, 'conflict': 0.0437}}
{'aspect_term': 'productivity', 'predicted_polarity': 'negative', 'confidence': 0.4669, 'all_class_scores': {'negative': 0.4669, 'neutral': 0.0257, 'positive': 0.4545, 'conflict': 0.0529}}
{'aspect_term': '8gb ram', 'predicted_polarity': 'negative', 'confidence': 0.9131, 'all_class_scores': {'negative': 0.9131, 'neutral': 0.0121, 'positive': 0.0329, 'conflict': 0.042}}
{'aspect_term': 'gaming performance', 'predicted_polarity': 'negative', 'confidence': 0.7303, 'all_class_scores': {'negative': 0.7303, 'neutral': 0.0248, 'positive': 0.112, 'conflict': 0.1329}}
